# Spark practice: путь преобразования данных

Сквозной пример: Raw -> чтение -> выбор колонок -> типизация -> очистка -> дедупликация -> join -> агрегация -> проверки -> Stage/Mart. Все операции находятся внутри одного контекстного менеджера, поэтому DataFrame не используется после закрытия Spark.

## Конфигурация, DataFrame API, SQL и PipeSQL

В production чтение может выглядеть как `spark.table("raw.combat_events")`, `spark.read.parquet(path)` или `spark.read.json(path)`. PipeSQL показан в комментарии: его синтаксис зависит от версии Spark, а обычный Spark SQL остаётся переносимым эквивалентом.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark_config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.shuffle.partitions": "8",
    "spark.sql.session.timeZone": "UTC",
}

with SparkSessionContext("spark-practice-data-transformation", config=spark_config) as ctx:
    spark = ctx.spark

    # RAW: строки, NULL, дубликат и неожиданный тип события.
    raw_events = spark.createDataFrame([
        ("e1", 101, 7, "damage", "2026-01-01 12:10:00", "125", None, 20, 1),
        ("e2", 101, 7, "kill", "2026-01-01 12:11:00", "300", "5", 30, 2),
        ("e2", 101, 7, "kill", "2026-01-01 12:11:00", "300", "5", 30, 3),
        ("e3", 102, 12, "damage", "2026-01-02 08:00:00", "unknown", None, 10, 4),
        ("e4", 102, None, "unexpected", None, None, "2", 0, 5),
    ], ["event_id", "match_id", "hero_id", "event_type", "event_time", "damage", "healing", "gold", "loaded_at"])
    raw_events.createOrReplaceTempView("raw_combat_events")
    raw_events.printSchema()
    raw_events.show(10, truncate=False)
    print(raw_events.columns)
    spark.sql("DESCRIBE raw_combat_events").show(truncate=False)

    # Stage: только нужные поля, явная типизация, NULL и категории.
    events = (raw_events
        .select("event_id", "match_id", "hero_id", "event_type", "event_time", "damage", "healing", "gold", "loaded_at")
        .withColumn("event_time", F.to_timestamp("event_time"))
        .withColumn("hero_id", F.col("hero_id").cast("int"))
        .withColumn("damage", F.col("damage").cast("double"))
        .withColumn("healing", F.col("healing").cast("double"))
        .withColumn("gold", F.col("gold").cast("double"))
        .withColumn("event_date", F.to_date("event_time"))
        .withColumn("damage_value", F.coalesce(F.col("damage"), F.lit(0.0)))
        .withColumn("event_group", F.when(F.col("event_type").isin("damage", "kill"), "combat").when(F.col("event_type") == "healing", "support").otherwise("other"))
        .filter(F.col("event_id").isNotNull() & F.col("hero_id").isNotNull())
    )
    latest = Window.partitionBy("event_id").orderBy(F.col("loaded_at").desc())
    stg_events = events.withColumn("row_number", F.row_number().over(latest)).filter(F.col("row_number") == 1).drop("row_number")
    stg_events.createOrReplaceTempView("stg_combat_events")
    stg_events.show(truncate=False)

    # SQL: фильтр и агрегация меняют grain с события на героя за день.
    mart_sql = spark.sql("""
        SELECT event_date, hero_id, SUM(damage_value) AS damage_total,
               COUNT(*) AS events_count, COUNT(event_id) AS event_ids_count,
               COUNT(DISTINCT event_id) AS distinct_events_count, AVG(damage) AS damage_avg
        FROM stg_combat_events
        WHERE event_type IN ('damage', 'kill') AND damage_value > 0
        GROUP BY event_date, hero_id
        ORDER BY event_date, hero_id
    """)
    mart_sql.show()

    # PipeSQL-идея (Spark 4+): FROM ... |> WHERE ... |> AGGREGATE ...
    # pipe_result = spark.sql("""
    #   FROM stg_combat_events
    #   |> WHERE event_type IN ('damage', 'kill')
    #   |> AGGREGATE SUM(damage_value) AS damage_total, COUNT(*) AS events_count
    #      GROUP BY event_date, hero_id
    # """)

    # Join со справочником и window: строки сохраняются, добавляется rank.
    heroes = spark.createDataFrame([(7, "Axe"), (12, "Juggernaut")], ["hero_id", "hero_name"])
    events_with_heroes = stg_events.join(heroes, on="hero_id", how="left")
    top_window = Window.partitionBy("match_id", "hero_id").orderBy(F.col("damage_value").desc())
    top_events = events_with_heroes.withColumn("rank", F.row_number().over(top_window)).filter(F.col("rank") == 1)
    mart = (events_with_heroes.filter(F.col("event_type").isin("damage", "kill"))
        .groupBy("event_date", "hero_id", "hero_name")
        .agg(F.sum("damage_value").alias("damage_total"), F.count("*").alias("events_count")))
    top_events.show()
    mart.printSchema()
    mart.show()

    # Проверки grain, обязательных значений и диапазонов перед записью.
    assert mart.filter(F.col("hero_id").isNull()).count() == 0
    assert mart.filter(F.col("events_count") <= 0).count() == 0
    assert mart.select("event_date", "hero_id").count() == mart.select("event_date", "hero_id").distinct().count()
    # mart.writeTo("stage.combat_events").createOrReplace()
    # mart.writeTo("mart.hero_damage_daily").createOrReplace()
    print("Проверки пройдены; Spark-контекст будет закрыт автоматически.")
